# Nepali Sign Language (NSL) Alphabet Recognition — Full Pipeline

**CSC60904 Deep Learning · Group Assignment · Track T3 · "AI for the Himalayas 2026"**

This single notebook runs the complete supervised deep-learning pipeline end to end:

`data audit / cleaning → EDA → stratified split → augmentation → 2 models → evaluation → error analysis → comparison → saved artifacts`

It implements **two architectures** (as required): a from-scratch **Baseline CNN** and a **MobileNetV2 transfer-learning** model (feature-extraction + fine-tuning), and applies several **regularisation/optimisation** techniques (batch-norm, dropout, L2, early stopping, LR scheduling, data augmentation).

**Dataset:** *Nepali Sign Language Character Dataset* (Poudel et al., 2025) — 36 character classes, ~1,000 images per class, in two versions: **Plain Background** and **Random Background** (MIT licensed). There is also a peer-reviewed benchmark on this exact data (Poudel et al., *KEC Journal of Science and Engineering*, 2026) reporting **MobileNetV2 ≈ 90.45%** and **ResNet50 ≈ 88.78%** — a good target to compare your results against and a citable source for your Literature Review.

---

### How to run
1. `pip install tensorflow scikit-learn pandas matplotlib pillow` (GPU build of TensorFlow for your CUDA/cuDNN).
2. Download the dataset from Kaggle and unzip `NSL.zip` so you have:
   ```
   <DATA_ROOT>/
     Plain Background/   0 1 2 ... 35   (1000 imgs each)
     Random Background/  0 1 2 ... 35   (1000 imgs each)
   ```
3. Set `DATA_ROOT` in the **Configuration** cell.
4. (Optional first pass) set `subset_fraction = 0.1` to smoke-test the whole flow in a few minutes, then set it back to `1.0` for the real run.
5. Run all cells top to bottom.

### Suggested split of work (3 members)
- **Data Engineer** — Config, Data Audit & Cleaning, EDA, Split, Pipeline (Sec. 1–5) + dataset ethics/consent notes.
- **ML Engineer** — Baseline CNN + Transfer Learning, hyperparameter tuning (Sec. 6–7).
- **Evaluation & Ethics Lead** — Evaluation, Error Analysis, Comparison, demo, ethics section (Sec. 8–10).

> Keep everyone committing to the shared GitHub repo — commit history from all members is explicitly checked in the rubric.


## 0. Environment setup & GPU check

In [ ]:
import os, sys, glob, random, hashlib, json, time, math, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

warnings.filterwarnings("ignore", category=UserWarning)

print("Python      :", sys.version.split()[0])
print("TensorFlow  :", tf.__version__)
print("Keras       :", keras.__version__)
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("\nGPUs detected:", gpus)
if gpus:
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
        print("Memory growth enabled on", len(gpus), "GPU(s).")
    except RuntimeError as e:
        print("Could not set memory growth:", e)
else:
    print("No GPU found -> training will run on CPU (much slower).")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Configuration

Everything you might want to tweak lives here. **Set `DATA_ROOT` to your local dataset path.**

In [ ]:
from dataclasses import dataclass

# --- Locate the repo root so paths work whether you launch this notebook
#     from the repo root OR from inside the /notebooks folder. ---
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

# >>> EDIT the dataset folder name if needed ("NSLv2" vs "NSL") <<<
DATA_ROOT   = REPO_ROOT / "data" / "NSLv2"
MODELS_DIR  = REPO_ROOT / "models"      # saved .keras models + label_map.json
RESULTS_DIR = REPO_ROOT / "results"     # figures, metrics, logs, comparison csv
FIG_DIR     = RESULTS_DIR / "figures"

for d in (MODELS_DIR, RESULTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

@dataclass
class CFG:
    data_root: Path = DATA_ROOT
    backgrounds: tuple = ("Plain Background", "Random Background")  # subsets to include
    img_size: tuple = (224, 224)      # MobileNetV2 native input size
    batch_size: int = 32
    val_frac: float = 0.15
    test_frac: float = 0.15
    seed: int = SEED
    baseline_epochs: int = 15
    tl_epochs: int = 10               # transfer-learning: feature-extraction phase
    ft_epochs: int = 5                # transfer-learning: fine-tuning phase
    subset_fraction: float = 1.0      # set < 1.0 (e.g. 0.1) for a quick smoke test

cfg = CFG()
print("Repo root :", REPO_ROOT)
print("Data root :", cfg.data_root, "| exists:", cfg.data_root.exists())
print("Models -> ", MODELS_DIR)
print("Results ->", RESULTS_DIR)

## 2. Data audit & cleaning

Before any modelling we (a) build a manifest of every image, (b) check for **corrupt/unreadable** files, (c) check for **exact duplicates**, and (d) audit **image size / colour mode**. This is what turns "it works on my machine" into a reproducible, documented pipeline (rubric: *Data Pipeline* and *Data* section of the report).

In [ ]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

def scan_dataset(root: Path, backgrounds):
    rows = []
    for bg in backgrounds:
        bg_dir = root / bg
        if not bg_dir.exists():
            print(f"[warn] missing background folder: {bg_dir}")
            continue
        for label_dir in sorted([p for p in bg_dir.iterdir() if p.is_dir()],
                                key=lambda p: (len(p.name), p.name)):
            label = label_dir.name
            for ext in IMG_EXTS:
                for fp in label_dir.glob(f"*{ext}"):
                    rows.append({"filepath": str(fp), "label": label, "background": bg})
    return pd.DataFrame(rows)

df = scan_dataset(cfg.data_root, cfg.backgrounds)
assert len(df) > 0, "No images found. Check DATA_ROOT and the folder names in cfg.backgrounds."
df["label"] = df["label"].astype(str)
print("Total images found :", len(df))
print("Unique labels      :", df["label"].nunique())
print("Backgrounds        :", sorted(df["background"].unique()))
df.head()

In [ ]:
# (b) Integrity check: find corrupt / unreadable images.
# im.verify() checks the file header quickly without a full decode.
def verify_images(paths):
    bad = []
    for p in paths:
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception as e:
            bad.append((p, str(e)))
    return bad

# For speed we check a random sample. Set to df["filepath"].tolist() for a full audit.
check_paths = random.sample(df["filepath"].tolist(), min(3000, len(df)))
bad = verify_images(check_paths)
print(f"Corrupt/unreadable images in sample of {len(check_paths)}: {len(bad)}")
for p, e in bad[:10]:
    print("   ", p, "->", e)

if bad:
    bad_paths = {p for p, _ in bad}
    df = df[~df["filepath"].isin(bad_paths)].reset_index(drop=True)
    print("Dropped", len(bad_paths), "broken images. Remaining:", len(df))

In [ ]:
# (c) Exact-duplicate detection via MD5 (sampled for speed).
def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

hash_sample = df.sample(min(3000, len(df)), random_state=SEED).copy()
hash_sample["md5"] = hash_sample["filepath"].map(file_md5)
dupe_mask = hash_sample["md5"].duplicated(keep=False)
print("Exact-duplicate files in sample:", int(dupe_mask.sum()))
if dupe_mask.any():
    display(hash_sample[dupe_mask].sort_values("md5").head(10))

In [ ]:
# (d) Image size / colour-mode audit on a sample.
def image_meta(path):
    with Image.open(path) as im:
        return im.size[0], im.size[1], im.mode

meta = df.sample(min(500, len(df)), random_state=SEED).copy()
meta[["width", "height", "mode"]] = pd.DataFrame(
    meta["filepath"].map(image_meta).tolist(), index=meta.index)
print(meta[["width", "height"]].describe().round(1))
print("\nColour modes:", dict(Counter(meta["mode"])))

In [ ]:
# Folder-number -> Nepali character mapping (Devanagari, romanised).
# NOTE: This follows the conventional Devanagari consonant order and is a TEMPLATE.
# Verify the exact folder->character mapping against the dataset's own reference
# table before quoting it in your report.
NEPALI_CHAR_MAP = {
    0:("\u0915","ka"),  1:("\u0916","kha"), 2:("\u0917","ga"),  3:("\u0918","gha"), 4:("\u0919","nga"),
    5:("\u091a","cha"), 6:("\u091b","chha"),7:("\u091c","ja"),  8:("\u091d","jha"), 9:("\u091e","yna"),
    10:("\u091f","Ta"), 11:("\u0920","Tha"),12:("\u0921","Da"), 13:("\u0922","Dha"),14:("\u0923","Na"),
    15:("\u0924","ta"), 16:("\u0925","tha"),17:("\u0926","da"), 18:("\u0927","dha"),19:("\u0928","na"),
    20:("\u092a","pa"), 21:("\u092b","pha"),22:("\u092c","ba"), 23:("\u092d","bha"),24:("\u092e","ma"),
    25:("\u092f","ya"), 26:("\u0930","ra"), 27:("\u0932","la"), 28:("\u0935","wa"), 29:("\u0936","sha"),
    30:("\u0937","Sha"),31:("\u0938","sa"), 32:("\u0939","ha"),
    33:("\u0915\u094d\u0937","ksha"), 34:("\u0924\u094d\u0930","tra"), 35:("\u091c\u094d\u091e","gya"),
}

def deva(label):
    try:    return NEPALI_CHAR_MAP.get(int(label), (str(label), ""))[0]
    except (ValueError, TypeError): return str(label)

def roman(label):
    try:    return NEPALI_CHAR_MAP.get(int(label), ("", str(label)))[1] or str(label)
    except (ValueError, TypeError): return str(label)

print("Example mapping -> folder 0 =", deva("0"), "(", roman("0"), ")")

## 3. Exploratory Data Analysis (EDA)

Class balance, sample images, background comparison, and dimension audit.

> **Note on fonts:** matplotlib's default fonts do not include Devanagari glyphs, so plot labels use the **romanised** names to avoid empty boxes. Devanagari is preserved in the saved `label_map.json` and in printed output. To show Devanagari in plots, register a Devanagari-capable font (e.g. *Noto Sans Devanagari*).

In [ ]:
CLASS_NAMES = sorted(df["label"].unique(), key=lambda x: int(x))
NUM_CLASSES = len(CLASS_NAMES)
label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
idx_to_label = {i: name for name, i in label_to_idx.items()}
roman_names  = [roman(idx_to_label[i]) for i in range(NUM_CLASSES)]
print("NUM_CLASSES:", NUM_CLASSES)

In [ ]:
# Class distribution (overall) — should be ~balanced (~1000/class per background).
overall = df["label"].value_counts().reindex(CLASS_NAMES)
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar([roman(l) for l in CLASS_NAMES], overall.values)
ax.set_title("Image count per class (all selected backgrounds)")
ax.set_ylabel("count"); plt.xticks(rotation=90); plt.tight_layout()
plt.savefig(FIG_DIR/"class_distribution.png", dpi=120); plt.show()

print("min/median/max per class:",
      int(overall.min()), int(overall.median()), int(overall.max()))

In [ ]:
# Class distribution split by background.
if df["background"].nunique() > 1:
    pivot = (df.pivot_table(index="label", columns="background",
                            values="filepath", aggfunc="count")
               .reindex(CLASS_NAMES))
    pivot.index = [roman(l) for l in CLASS_NAMES]
    pivot.plot(kind="bar", figsize=(14, 4), title="Image count per class by background")
    plt.tight_layout(); plt.show()
    print(df.groupby("background").size())

In [ ]:
# One sample image per class.
def load_rgb(path, size=None):
    with Image.open(path) as im:
        im = im.convert("RGB")
        if size: im = im.resize(size)
        return np.asarray(im)

n = NUM_CLASSES
cols = 6; rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2*cols, 2*rows))
for ax, cls in zip(axes.ravel(), CLASS_NAMES):
    ax.imshow(load_rgb(df[df["label"] == cls]["filepath"].iloc[0], (128, 128)))
    ax.set_title(roman(cls), fontsize=9); ax.axis("off")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.suptitle("One sample per class (romanised labels)", y=1.0)
plt.tight_layout()
plt.savefig(FIG_DIR/"class_samples.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# Plain vs Random background — same class, side by side (great for the report).
if df["background"].nunique() > 1:
    compare = CLASS_NAMES[:4]
    fig, axes = plt.subplots(len(compare), 2, figsize=(6, 3*len(compare)))
    for i, cls in enumerate(compare):
        for j, bg in enumerate(["Plain Background", "Random Background"]):
            sub = df[(df.label == cls) & (df.background == bg)]
            axes[i, j].axis("off")
            if len(sub):
                axes[i, j].imshow(load_rgb(sub.filepath.iloc[0], (128, 128)))
            axes[i, j].set_title(f"{roman(cls)} - {bg}", fontsize=9)
    plt.tight_layout()
    plt.savefig(FIG_DIR/"background_compare.png", dpi=120, bbox_inches="tight")
    plt.show()

## 4. Stratified train / validation / test split

We split **by file** with stratification on the label so every class is proportionally represented in each split (default 70 / 15 / 15). We also assert there is **no leakage** between train and test.

In [ ]:
work = df.copy()
if cfg.subset_fraction < 1.0:
    work = (work.groupby("label", group_keys=False)
                .apply(lambda g: g.sample(frac=cfg.subset_fraction, random_state=SEED)))
    print(f"Using subset ({cfg.subset_fraction:.0%}):", len(work), "images")

train_df, temp_df = train_test_split(
    work, test_size=cfg.val_frac + cfg.test_frac,
    stratify=work["label"], random_state=SEED)

rel_test = cfg.test_frac / (cfg.val_frac + cfg.test_frac)
val_df, test_df = train_test_split(
    temp_df, test_size=rel_test, stratify=temp_df["label"], random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(d):7d} images  ({len(d)/len(work):.1%})")

# Leakage check
assert set(train_df.filepath) & set(test_df.filepath) == set()
assert set(train_df.filepath) & set(val_df.filepath)  == set()
print("No leakage between splits. OK")

## 5. Input pipeline (`tf.data`) + augmentation

Images are decoded, resized and kept in `[0, 255]` float; each model rescales internally (baseline -> `[0,1]`, MobileNetV2 -> `[-1,1]`).

**Domain-aware augmentation:** we deliberately **do not** use horizontal flips — mirroring a hand can turn one sign into a different (or invalid) sign, and the dataset is right-hand. We use small rotations, zoom, translation, brightness and contrast jitter to simulate real webcam variation. Augmentation is applied to the **training set only**.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(frame, shuffle=False):
    paths  = frame["filepath"].values
    labels = frame["label"].map(label_to_idx).values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, cfg.img_size)
        img = tf.cast(img, tf.float32)          # keep [0,255]; models rescale internally
        return img, label
    return ds.map(_load, num_parallel_calls=AUTOTUNE)

# NOTE: no RandomFlip on purpose (see markdown above).
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.06, 0.06),
    layers.RandomContrast(0.10),
    layers.RandomBrightness(0.10, value_range=(0, 255)),
], name="augment")

def prepare(ds, training=False):
    ds = ds.batch(cfg.batch_size)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = prepare(make_ds(train_df, shuffle=True), training=True)
val_ds   = prepare(make_ds(val_df))
test_ds  = prepare(make_ds(test_df))
print("Pipelines built.")

In [ ]:
# Visualise one augmented training batch.
imgs, labs = next(iter(train_ds))
plt.figure(figsize=(9, 9))
for i in range(min(9, imgs.shape[0])):
    plt.subplot(3, 3, i + 1)
    plt.imshow(tf.cast(tf.clip_by_value(imgs[i], 0, 255), tf.uint8).numpy())
    plt.title(roman(idx_to_label[int(labs[i])])); plt.axis("off")
plt.suptitle("Augmented training batch"); plt.tight_layout(); plt.show()

## 6. Model 1 — Baseline CNN (from scratch)

A compact VGG-style CNN. Regularisation baked in: **Batch Normalisation**, **Dropout**, **L2** weight decay, and (in training) **early stopping** + **LR scheduling**. This is your "beyond-MLP" baseline that the transfer-learning model must beat.

In [ ]:
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inputs)
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

baseline = build_baseline_cnn(cfg.img_size + (3,), NUM_CLASSES)
baseline.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
baseline.summary()

In [ ]:
def callbacks_for(tag):
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=2, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint(str(MODELS_DIR/f"{tag}_best.keras"),
                                        monitor="val_accuracy", save_best_only=True),
        keras.callbacks.CSVLogger(str(RESULTS_DIR/f"{tag}_log.csv")),
    ]

t0 = time.time()
hist_base = baseline.fit(train_ds, validation_data=val_ds,
                         epochs=cfg.baseline_epochs, callbacks=callbacks_for("baseline"))
baseline_time = time.time() - t0
print(f"Baseline training time: {baseline_time/60:.1f} min")

In [ ]:
def plot_history(hist, title, save=None):
    h = hist.history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h["loss"], label="train"); ax[0].plot(h["val_loss"], label="val")
    ax[0].set_title(f"{title} - loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h["accuracy"], label="train"); ax[1].plot(h["val_accuracy"], label="val")
    ax[1].set_title(f"{title} - accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout()
    if save: plt.savefig(save, dpi=120)
    plt.show()

plot_history(hist_base, "Baseline CNN", FIG_DIR/"baseline_curves.png")

## 7. Model 2 — MobileNetV2 transfer learning

Two phases (standard best practice):
1. **Feature extraction** — freeze the ImageNet-pretrained backbone, train only the new classifier head.
2. **Fine-tuning** — unfreeze the top layers and continue at a much lower learning rate (BatchNorm layers kept frozen).

MobileNetV2 is lightweight, which matters for the eventual Nepal deployment story (low-compute, potentially on-device).

In [ ]:
def build_mobilenet(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1./127.5, offset=-1)(inputs)   # MobileNetV2 expects [-1, 1]
    base = keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                          input_tensor=x)
    base.trainable = False
    y = layers.GlobalAveragePooling2D()(base.output)
    y = layers.Dropout(0.3)(y)
    outputs = layers.Dense(num_classes, activation="softmax")(y)
    return keras.Model(inputs, outputs, name="mobilenetv2_tl"), base

tl_model, base = build_mobilenet(cfg.img_size + (3,), NUM_CLASSES)
tl_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Trainable params (feature-extraction phase):",
      f"{np.sum([np.prod(v.shape) for v in tl_model.trainable_variables]):,}")

In [ ]:
# Phase 1: feature extraction
t0 = time.time()
hist_fe = tl_model.fit(train_ds, validation_data=val_ds,
                       epochs=cfg.tl_epochs, callbacks=callbacks_for("mobilenet_fe"))
plot_history(hist_fe, "MobileNetV2 (feature extraction)",
             FIG_DIR/"tl_fe_curves.png")

In [ ]:
# Phase 2: fine-tuning (unfreeze top ~30 layers, keep BatchNorm frozen, low LR)
base.trainable = True
FREEZE_UNTIL = len(base.layers) - 30
for layer in base.layers[:FREEZE_UNTIL]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

tl_model.compile(optimizer=keras.optimizers.Adam(1e-5),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Trainable params (fine-tuning phase):",
      f"{np.sum([np.prod(v.shape) for v in tl_model.trainable_variables]):,}")

hist_ft = tl_model.fit(train_ds, validation_data=val_ds,
                       epochs=cfg.ft_epochs, callbacks=callbacks_for("mobilenet_ft"))
tl_time = time.time() - t0
plot_history(hist_ft, "MobileNetV2 (fine-tuning)",
             FIG_DIR/"tl_ft_curves.png")

## 8. Evaluation & error analysis

Task-appropriate metrics (accuracy, macro precision/recall/F1), **confusion matrix**, **per-class breakdown**, and concrete **misclassified examples** with the model's confidence. Because `test_ds` is built from `test_df` in order (no shuffle), predictions line up with `test_df` rows, so we can pull the actual failing images.

In [ ]:
def get_preds(model, ds):
    y_true, y_prob = [], []
    for xb, yb in ds:
        y_true.append(yb.numpy())
        y_prob.append(model.predict(xb, verbose=0))
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    return y_true, y_prob.argmax(1), y_prob

def evaluate_model(model, ds, tag):
    y_true, y_pred, y_prob = get_preds(model, ds)
    acc = float((y_true == y_pred).mean())
    print(f"=== {tag} ===  test accuracy: {acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=roman_names,
                                digits=3, zero_division=0))
    return y_true, y_pred, y_prob, acc

def plot_cm(y_true, y_pred, tag, save=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(roman_names, rotation=90, fontsize=7)
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(roman_names, fontsize=7)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Confusion matrix - {tag}")
    fig.colorbar(im, fraction=0.046, pad=0.04)
    plt.tight_layout()
    if save: plt.savefig(save, dpi=120)
    plt.show()
    return cm

In [ ]:
# Baseline CNN
yb_true, yb_pred, yb_prob, acc_base = evaluate_model(baseline, test_ds, "Baseline CNN")
cm_base = plot_cm(yb_true, yb_pred, "Baseline CNN", FIG_DIR/"cm_baseline.png")

In [ ]:
# MobileNetV2 transfer learning
yt_true, yt_pred, yt_prob, acc_tl = evaluate_model(tl_model, test_ds, "MobileNetV2 (TL+FT)")
cm_tl = plot_cm(yt_true, yt_pred, "MobileNetV2 (TL+FT)", FIG_DIR/"cm_mobilenet.png")

In [ ]:
# Per-class accuracy (MobileNetV2)
def per_class_accuracy(cm):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.nan_to_num(cm.diagonal() / cm.sum(axis=1))

pca = per_class_accuracy(cm_tl)
order = np.argsort(pca)
plt.figure(figsize=(14, 4))
plt.bar([roman_names[i] for i in order], pca[order])
plt.axhline(acc_tl, color="crimson", ls="--", label=f"overall {acc_tl:.3f}")
plt.ylabel("accuracy"); plt.title("Per-class accuracy (MobileNetV2), worst -> best")
plt.xticks(rotation=90); plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/"per_class_acc.png", dpi=120); plt.show()

print("Weakest 5 classes:", [roman_names[i] for i in order[:5]])

In [ ]:
# Most-confused class pairs (where the model most often mixes up two signs).
def top_confused_pairs(cm, k=10):
    c = cm.copy().astype(float); np.fill_diagonal(c, 0)
    rows = []
    for _ in range(k):
        i, j = np.unravel_index(np.argmax(c), c.shape)
        if c[i, j] == 0: break
        rows.append({"true": roman_names[i], "predicted_as": roman_names[j],
                     "count": int(cm[i, j])})
        c[i, j] = 0
    return pd.DataFrame(rows)

top_confused_pairs(cm_tl)

In [ ]:
# Show actual misclassified images (true vs predicted + confidence).
test_paths = test_df["filepath"].values

def show_misclassified(y_true, y_pred, y_prob, paths, tag, n=12):
    wrong = np.where(y_true != y_pred)[0]
    if len(wrong) == 0:
        print("No misclassifications."); return
    pick = wrong[:n]
    cols = 4; rows = int(np.ceil(len(pick) / cols))
    plt.figure(figsize=(4*cols, 3.4*rows))
    for i, idx in enumerate(pick):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(load_rgb(paths[idx], (128, 128)))
        plt.title(f"true {roman(idx_to_label[int(y_true[idx])])}  |  "
                  f"pred {roman(idx_to_label[int(y_pred[idx])])} ({y_prob[idx].max():.2f})",
                  fontsize=9, color="crimson")
        plt.axis("off")
    plt.suptitle(f"Misclassified examples - {tag}"); plt.tight_layout()
    plt.savefig(FIG_DIR/f"errors_{tag}.png", dpi=120, bbox_inches="tight")
    plt.show()

show_misclassified(yt_true, yt_pred, yt_prob, test_paths, "MobileNetV2")

In [ ]:
# (Optional) Grad-CAM: what is the MobileNetV2 model "looking at"?
# Wrapped in try/except so a version quirk never breaks the run.
def last_conv_name(model):
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer.name
    return None

def gradcam(model, img_batch, conv_name):
    grad_model = keras.Model(model.inputs,
                             [model.get_layer(conv_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch)
        cls = tf.argmax(preds[0])
        loss = preds[:, cls]
    grads = tape.gradient(loss, conv_out)[0]
    conv_out = conv_out[0]
    weights = tf.reduce_mean(grads, axis=(0, 1))
    heat = tf.nn.relu(tf.reduce_sum(conv_out * weights, axis=-1))
    heat = heat / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy(), int(cls)

try:
    conv_name = last_conv_name(tl_model)
    sample_path = test_df["filepath"].iloc[0]
    arr = load_rgb(sample_path, tuple(cfg.img_size)).astype("float32")[None, ...]
    heat, cls = gradcam(tl_model, arr, conv_name)
    heat_up = np.array(Image.fromarray(np.uint8(255*heat)).resize(cfg.img_size)) / 255.0
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(arr[0].astype("uint8")); ax[0].set_title("input"); ax[0].axis("off")
    ax[1].imshow(arr[0].astype("uint8")); ax[1].imshow(heat_up, cmap="jet", alpha=0.45)
    ax[1].set_title(f"Grad-CAM -> {roman(idx_to_label[cls])}"); ax[1].axis("off")
    plt.tight_layout(); plt.savefig(FIG_DIR/"gradcam.png", dpi=120); plt.show()
except Exception as e:
    print("Grad-CAM skipped:", e)

## 9. Model comparison

Side-by-side comparison of the two architectures. Compare your MobileNetV2 number against the published benchmark (**≈90.45%**) in your report and discuss *why* transfer learning wins here (ImageNet features + limited data).

In [ ]:
comparison = pd.DataFrame([
    {"model": "Baseline CNN",
     "params": baseline.count_params(),
     "test_accuracy": acc_base,
     "macro_f1": f1_score(yb_true, yb_pred, average="macro"),
     "train_time_min": baseline_time / 60},
    {"model": "MobileNetV2 (TL+FT)",
     "params": tl_model.count_params(),
     "test_accuracy": acc_tl,
     "macro_f1": f1_score(yt_true, yt_pred, average="macro"),
     "train_time_min": tl_time / 60},
])
comparison_display = comparison.copy()
comparison_display["params"] = comparison_display["params"].map(lambda x: f"{x:,}")
comparison_display = comparison_display.round(4)
comparison_display

In [ ]:
ax = comparison.plot(x="model", y=["test_accuracy", "macro_f1"], kind="bar",
                     figsize=(7, 4), title="Model comparison")
ax.set_ylabel("score"); plt.xticks(rotation=0); plt.ylim(0, 1)
plt.tight_layout(); plt.savefig(FIG_DIR/"model_comparison.png", dpi=120); plt.show()

## 10. Save artifacts & next steps

In [ ]:
# Persist final models, label map, and metrics for the repo / demo.
baseline.save(MODELS_DIR/"baseline_cnn.keras")
tl_model.save(MODELS_DIR/"mobilenetv2_tl.keras")

label_map = {str(i): {"folder": idx_to_label[i],
                      "devanagari": deva(idx_to_label[i]),
                      "roman": roman(idx_to_label[i])}
             for i in range(NUM_CLASSES)}
with open(MODELS_DIR/"label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

comparison.to_csv(RESULTS_DIR/"model_comparison.csv", index=False)
with open(RESULTS_DIR/"metrics.json", "w") as f:
    json.dump({"baseline_test_acc": acc_base, "mobilenet_test_acc": acc_tl,
               "num_classes": NUM_CLASSES, "n_train": len(train_df),
               "n_val": len(val_df), "n_test": len(test_df)}, f, indent=2)

print("Saved models to :", MODELS_DIR)
print("   - baseline_cnn.keras, mobilenetv2_tl.keras, label_map.json")
print("Saved results to:", RESULTS_DIR)
print("   - model_comparison.csv, metrics.json, figures/*.png, *_log.csv")

## 11. In-notebook interactive demo (Jupyter)

A lightweight demo that runs inside this notebook (satisfies the "Jupyter-based interactive demo" option). Upload a hand-sign image or classify a random test image and see the top-3 predictions. The standalone Streamlit `app.py` is still provided separately for a webcam UI.

In [ ]:
# Prediction helpers for the demo.
def predict_pil(pil_img, model, k=3):
    arr = np.asarray(pil_img.convert("RGB").resize(cfg.img_size)).astype("float32")[None, ...]
    probs = model.predict(arr, verbose=0)[0]
    top = probs.argsort()[::-1][:k]
    return [(roman(idx_to_label[int(i)]), deva(idx_to_label[int(i)]), float(probs[i])) for i in top]

def show_prediction(pil_img, model, title=""):
    preds = predict_pil(pil_img, model)
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    ax[0].imshow(pil_img.convert("RGB").resize((160, 160))); ax[0].axis("off")
    ax[0].set_title(title or "input")
    names = [r for r, d, p in preds][::-1]
    vals  = [p for r, d, p in preds][::-1]
    ax[1].barh(names, vals); ax[1].set_xlim(0, 1); ax[1].set_title("top-3 confidence")
    for i, (r, d, p) in enumerate(preds[::-1]):
        ax[1].text(min(p + 0.02, 0.9), i, f"{p*100:.1f}%", va="center")
    plt.tight_layout(); plt.show()
    best = preds[0]
    print(f"Prediction: {best[1]}  ({best[0]})   confidence {best[2]*100:.1f}%")
    return preds

# Pick which trained model the demo should use:
demo_model = tl_model      # swap to `baseline` to compare

In [ ]:
# Classify a random test-set image (re-run this cell for a new sample).
rand_path = test_df.sample(1)["filepath"].iloc[0]
_ = show_prediction(Image.open(rand_path), demo_model, title="random test image")

In [ ]:
# Interactive upload demo. Needs ipywidgets:  pip install ipywidgets
# (In classic Jupyter also run: jupyter nbextension enable --py widgetsnbextension)
try:
    import io
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    uploader = widgets.FileUpload(accept="image/*", multiple=False)
    out = widgets.Output()

    def _on_upload(change):
        with out:
            clear_output()
            val = uploader.value
            if not val:
                return
            # Handle both ipywidgets v7 (dict) and v8 (tuple) value formats.
            item = list(val.values())[0] if isinstance(val, dict) else val[0]
            img = Image.open(io.BytesIO(bytes(item["content"])))
            show_prediction(img, demo_model, title="uploaded image")

    uploader.observe(_on_upload, names="value")
    print("Upload a hand-sign image to classify it:")
    display(uploader, out)
except Exception as e:
    print("ipywidgets unavailable - use the function directly instead, e.g.:")
    print("    show_prediction(Image.open('path/to/sign.jpg'), demo_model)")
    print("Reason:", e)

### Demo (`app.py`)

A minimal **Streamlit** demo is provided as a separate `app.py` in this repo. After training, run:

```bash
pip install streamlit opencv-python
streamlit run app.py
```

It loads `artifacts/models/mobilenetv2_tl.keras` + `artifacts/label_map.json` and lets a non-technical user **upload an image or use their webcam** to get the predicted Nepali character with top-3 confidences. That satisfies the "lightweight demo interface" requirement.

---

### How this maps to the rubric
- **Data pipeline** — manifest, corruption/duplicate/size audit, stratified split, leakage check, augmentation (Sec. 2, 4, 5).
- **Two architectures + justification** — baseline CNN vs MobileNetV2 transfer learning (Sec. 6, 7).
- **Optimisation/regularisation (need ≥2, we use 6)** — BatchNorm, Dropout, L2, EarlyStopping, ReduceLROnPlateau, augmentation.
- **Evaluation** — metrics, confusion matrix, per-class breakdown, most-confused pairs, real misclassified examples, optional Grad-CAM (Sec. 8).
- **Comparison** — table + chart, benchmarked against published results (Sec. 9).
- **Demo** — in-notebook interactive demo (Sec. 11) + standalone Streamlit `app.py`.

### Things to still do yourselves (learning + report marks)
1. **Verify the folder->character mapping** against the dataset's reference before quoting it.
2. Add a **third architecture** (e.g. EfficientNetB0 or ResNet50) if you want to push into the top band — the code generalises easily.
3. Write the **ethics section** (bias by signer/skin-tone/lighting, cost of a wrong prediction for a Deaf user, digital divide, consent, carbon) — this is a hard gate in the rubric.
4. Capture a **small webcam test set** of your own to honestly measure the domain gap; that becomes a strong "identified challenge" for the Week 9 progress check.
